In [3]:
import pandas as pd
import numpy as np

print("🔄 Loading E-commerce Fraud Data...")

# Load the raw e-commerce transaction dataset
df_fraud = pd.read_csv("../data/raw/Fraud_Data.csv")

print(f"✅ Data loaded successfully. Shape: {df_fraud.shape[0]} rows, {df_fraud.shape[1]} columns\n")
print("--- 📋 Initial Data Types ---")
print(df_fraud.dtypes)

🔄 Loading E-commerce Fraud Data...
✅ Data loaded successfully. Shape: 151112 rows, 11 columns

--- 📋 Initial Data Types ---
user_id             int64
signup_time        object
purchase_time      object
purchase_value      int64
device_id          object
source             object
browser            object
sex                object
age                 int64
ip_address        float64
class               int64
dtype: object


In [4]:
print("--- 🔍 Checking for Missing Values ---")
missing_summary = df_fraud.isnull().sum()
print(missing_summary[missing_summary > 0] if missing_summary.sum() > 0 else "No missing values found!")

print("\n--- 👥 Checking for Duplicate Rows ---")
duplicate_rows = df_fraud.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

print("\n--- 🆔 Checking for Duplicate User IDs ---")
duplicate_users = df_fraud['user_id'].duplicated().sum()
print(f"Number of duplicate user_id records: {duplicate_users}")

--- 🔍 Checking for Missing Values ---
No missing values found!

--- 👥 Checking for Duplicate Rows ---
Number of duplicate rows: 0

--- 🆔 Checking for Duplicate User IDs ---
Number of duplicate user_id records: 0


In [5]:
print("🔄 Converting timestamp columns to datetime objects...")

# Convert time columns to proper datetime format
df_fraud['signup_time'] = pd.to_datetime(df_fraud['signup_time'])
df_fraud['purchase_time'] = pd.to_datetime(df_fraud['purchase_time'])

print("✅ Conversion complete. Updated datatypes:")
print(df_fraud[['signup_time', 'purchase_time']].dtypes)

🔄 Converting timestamp columns to datetime objects...
✅ Conversion complete. Updated datatypes:
signup_time      datetime64[ns]
purchase_time    datetime64[ns]
dtype: object


In [6]:
import pandas as pd
import numpy as np

print("🔄 Running all-in-one Geolocation Merge...")

# 1. Load the raw data files directly
df_fraud = pd.read_csv("../data/raw/Fraud_Data.csv")
df_ip = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

# 2. Fix the time columns format right away
df_fraud['signup_time'] = pd.to_datetime(df_fraud['signup_time'])
df_fraud['purchase_time'] = pd.to_datetime(df_fraud['purchase_time'])

# 3. Align data types to floats for clean numerical range matching
df_ip['lower_bound_ip_address'] = df_ip['lower_bound_ip_address'].astype(float)
df_ip['upper_bound_ip_address'] = df_ip['upper_bound_ip_address'].astype(float)
df_fraud['ip_address'] = df_fraud['ip_address'].astype(float)

# 4. Mandatory Sort for merge_asof lookup accuracy
df_ip = df_ip.sort_values('lower_bound_ip_address').reset_index(drop=True)
df_fraud = df_fraud.sort_values('ip_address').reset_index(drop=True)

# 5. Run the range-based lookup merge
df_merged = pd.merge_asof(
    df_fraud,
    df_ip,
    left_on='ip_address',
    right_on='lower_bound_ip_address',
    direction='backward'
)

# 6. Validate boundaries (if transaction IP exceeds upper bound, mark as Unknown)
out_of_bounds_mask = df_merged['ip_address'] > df_merged['upper_bound_ip_address']
df_merged.loc[out_of_bounds_mask, 'country'] = 'Unknown'
df_merged['country'] = df_merged['country'].fillna('Unknown')

# 7. Drop boundary tracking columns to match final required shape
df_cleaned_ecom = df_merged.drop(columns=['lower_bound_ip_address', 'upper_bound_ip_address'])

print(f"✅ Success! 'df_cleaned_ecom' is now safely created in memory.")
print(f"📊 Dataset Shape: {df_cleaned_ecom.shape[0]} rows, {df_cleaned_ecom.shape[1]} columns")

🔄 Running all-in-one Geolocation Merge...
✅ Success! 'df_cleaned_ecom' is now safely created in memory.
📊 Dataset Shape: 151112 rows, 12 columns


In [7]:
print("🔄 Calculating time duration from signup to purchase...")

# Calculate the difference in hours
df_cleaned_ecom['time_since_signup'] = (
    df_cleaned_ecom['purchase_time'] - df_cleaned_ecom['signup_time']
).dt.total_seconds() / 3600.0

print("✅ Velocity feature 'time_since_signup' created.")
print(df_cleaned_ecom[['signup_time', 'purchase_time', 'time_since_signup']].head())

🔄 Calculating time duration from signup to purchase...
✅ Velocity feature 'time_since_signup' created.
          signup_time       purchase_time  time_since_signup
0 2015-02-16 00:17:05 2015-03-08 10:00:39         489.726111
1 2015-03-08 04:03:22 2015-03-20 17:23:45         301.339722
2 2015-05-17 16:45:54 2015-05-26 08:54:34         208.144444
3 2015-03-03 19:58:39 2015-05-28 21:09:13        2065.176111
4 2015-03-20 00:31:27 2015-04-05 07:31:46         391.005278


In [8]:
print("🔄 Extracting hour of day and day of week features...")

# Extract hour (0-23) and day of week (0=Monday, 6=Sunday)
df_cleaned_ecom['hour_of_day'] = df_cleaned_ecom['purchase_time'].dt.hour
df_cleaned_ecom['day_of_week'] = df_cleaned_ecom['purchase_time'].dt.dayofweek

print("✅ Temporal features 'hour_of_day' and 'day_of_week' successfully added.")
print(df_cleaned_ecom[['purchase_time', 'hour_of_day', 'day_of_week']].head())

🔄 Extracting hour of day and day of week features...
✅ Temporal features 'hour_of_day' and 'day_of_week' successfully added.
        purchase_time  hour_of_day  day_of_week
0 2015-03-08 10:00:39           10            6
1 2015-03-20 17:23:45           17            4
2 2015-05-26 08:54:34            8            1
3 2015-05-28 21:09:13           21            3
4 2015-04-05 07:31:46            7            6


In [9]:
print("🔄 Calculating device and IP address transaction frequency profiles...")

# Count how many times each device_id appears across the dataset
df_cleaned_ecom['device_usage_count'] = df_cleaned_ecom.groupby('device_id')['device_id'].transform('count')

# Count how many times each ip_address appears across the dataset
df_cleaned_ecom['ip_usage_count'] = df_cleaned_ecom.groupby('ip_address')['ip_address'].transform('count')

print("✅ Behavioral velocity features successfully engineered.")
print(df_cleaned_ecom[['device_id', 'device_usage_count', 'ip_address', 'ip_usage_count']].head())

🔄 Calculating device and IP address transaction frequency profiles...
✅ Behavioral velocity features successfully engineered.
       device_id  device_usage_count     ip_address  ip_usage_count
0  ZCLZTAJPCRAQX                   1   52093.496895               1
1  YFGYOALADBHLT                   1   93447.138961               1
2  QZNVQTUITFTHH                   1  105818.501505               1
3  PIBUQMBIELMMG                   1  117566.664867               1
4  WFIIFCPIOGMHT                   1  131423.789042               1


In [10]:
print("💾 Saving engineered e-commerce dataset to processed folder...")

# Save to the designated processed folder path
output_path = "../data/processed/cleaned_fraud_data.csv"
df_cleaned_ecom.to_csv(output_path, index=False)

print(f"🎉 Success! Processed dataset saved to: {output_path}")
print(f"Final Data Frame Shape: {df_cleaned_ecom.shape}")

💾 Saving engineered e-commerce dataset to processed folder...
🎉 Success! Processed dataset saved to: ../data/processed/cleaned_fraud_data.csv
Final Data Frame Shape: (151112, 17)


In [11]:
print("🔄 One-hot encoding categorical variables and isolating the target...")

# Drop identifier columns that won't be used for pattern recognition
cols_to_drop = ['user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address']
df_modeling = df_cleaned_ecom.drop(columns=cols_to_drop)

# List of columns to explicitly encode
categorical_cols = ['source', 'browser', 'sex', 'country']

# Apply one-hot encoding
df_encoded = pd.get_dummies(df_modeling, columns=categorical_cols, drop_first=True)

# Separate features (X) from the target variable (y)
X = df_encoded.drop(columns=['class'])
y = df_encoded['class']

print(f"✅ Encoding complete.")
print(f"Total features for modeling: {X.shape[1]}")
print(f"Target distribution check:\n{y.value_counts()}")

🔄 One-hot encoding categorical variables and isolating the target...
✅ Encoding complete.
Total features for modeling: 195
Target distribution check:
class
0    136961
1     14151
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("🔄 Executing stratified train-test split...")

# 80/20 stratified split to preserve class proportions across subsets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("🔄 Normalizing numerical features with StandardScaler...")

# Select only numerical features that require scaling
numerical_cols = ['purchase_value', 'age', 'time_since_signup', 'hour_of_day', 'day_of_week', 'device_usage_count', 'ip_usage_count']

# Initialize the scaler
scaler = StandardScaler()

# Copy dataframes to avoid setting-with-copy warnings
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit only on the training data and transform both sets to prevent leakage
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

print(f"✅ Stratified split and scaling successful.")
print(f"Training features shape: {X_train_scaled.shape}, Test features shape: {X_test_scaled.shape}")

🔄 Executing stratified train-test split...
🔄 Normalizing numerical features with StandardScaler...
✅ Stratified split and scaling successful.
Training features shape: (120889, 195), Test features shape: (30223, 195)


In [13]:
from imblearn.over_sampling import SMOTE

print("📊 --- Class Distribution BEFORE Resampling ---")
print(f"Legitimate (Class 0): {np.bincount(y_train)[0]}")
print(f"Fraudulent (Class 1): {np.bincount(y_train)[1]}")

print("\n🔄 Applying SMOTE to the training split...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("\n📊 --- Class Distribution AFTER Resampling ---")
print(f"Legitimate (Class 0): {np.bincount(y_train_resampled)[0]}")
print(f"Fraudulent (Class 1): {np.bincount(y_train_resampled)[1]}")

print("\n📊 --- Verification: Test Split Remains Untouched ---")
print(f"Test Set Fraud Ratio: {y_test.mean() * 100:.3f}%")

📊 --- Class Distribution BEFORE Resampling ---
Legitimate (Class 0): 109568
Fraudulent (Class 1): 11321

🔄 Applying SMOTE to the training split...

📊 --- Class Distribution AFTER Resampling ---
Legitimate (Class 0): 109568
Fraudulent (Class 1): 109568

📊 --- Verification: Test Split Remains Untouched ---
Test Set Fraud Ratio: 9.364%


In [14]:
print("💾 Saving processed, scaled, and resampled training/testing sets...")

# Save matrices as compressed numpy arrays for ultra-fast loading later
np.savez_compressed('../data/processed/ecommerce_train_test.npz', 
                    X_train=X_train_resampled, 
                    X_test=X_test_scaled.values, 
                    y_train=y_train_resampled, 
                    y_test=y_test.values)

print("🎉 Success! All E-commerce components for Task 1 are completely finalized and stored.")

💾 Saving processed, scaled, and resampled training/testing sets...
🎉 Success! All E-commerce components for Task 1 are completely finalized and stored.
